# DS3/GSE182109 — sample-aware post-QC Scrublet diagnostic

This notebook reproduces the memory-conscious Scrublet screening recorded
for DS3.

Scrublet is run independently within each sample using:

- expected doublet rate: 0.05
- simulated doublet ratio: 1.0
- principal components: 20
- random seed: 0
- minimum sample size for checking: 500 cells
- maximum cells per run: 40,000

The historical run produced:

- 2,927 predicted doublets;
- 215,146 predicted singlets;
- 662 cells not checked because two samples contained fewer than 500
  post-QC cells.

These calls were retained as a diagnostic and were **not used to remove
cells from the main analysis**, whose DS3 input remained 218,735 cells.
This is required for consistency with the reported integrated total of
351,427 cells.

Input:

`data/processed/DS3_GSE182109/GSE182109_postQC.h5ad`

Output:

`data/processed/DS3_GSE182109/GSE182109_postQC_scrublet_checked.h5ad`

In [ ]:
import gc
import importlib.util
import inspect
import json
import os
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

print("Python:", sys.version)
print("scanpy:", sc.__version__)

In [ ]:
def find_repository_root(start: Path) -> Path:
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start


PROJECT_ROOT = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", find_repository_root(Path.cwd()))
).expanduser().resolve()

INPUT_H5AD = Path(
    os.environ.get(
        "GSE182109_POSTQC_H5AD",
        PROJECT_ROOT
        / "data"
        / "processed"
        / "DS3_GSE182109"
        / "GSE182109_postQC.h5ad",
    )
).expanduser().resolve()

OUTPUT_H5AD = Path(
    os.environ.get(
        "GSE182109_SCRUBLET_H5AD",
        PROJECT_ROOT
        / "data"
        / "processed"
        / "DS3_GSE182109"
        / "GSE182109_postQC_scrublet_checked.h5ad",
    )
).expanduser().resolve()

AUDIT_DIR = (
    PROJECT_ROOT
    / "results"
    / "qc"
    / "DS3_GSE182109"
    / "00c_doublet_diagnostic"
)
FIGURE_DIR = AUDIT_DIR / "figures"

EXPECTED_INPUT_SHAPE = (218_735, 30_284)
EXPECTED_PREDICTED_DOUBLETS = 2_927
EXPECTED_PREDICTED_SINGLETS = 215_146
EXPECTED_NOT_CHECKED = 662

SAMPLE_COLUMN = "sample"
MIN_CELLS_PER_SAMPLE = 500
MAX_CELLS_PER_RUN = 40_000
EXPECTED_DOUBLET_RATE = 0.05
SIM_DOUBLET_RATIO = 1.0
N_PCS = 20
RANDOM_SEED = 0

OUTPUT_H5AD.parent.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_H5AD.exists():
    raise FileNotFoundError(
        f"Post-QC input not found: {INPUT_H5AD}\n"
        "Run 00b_ds3_quality_control.ipynb first or set "
        "GSE182109_POSTQC_H5AD."
    )

if importlib.util.find_spec("scrublet") is None:
    raise ImportError(
        "The 'scrublet' package is required by scanpy.pp.scrublet. "
        "Install the repository environment before running this notebook."
    )

print("Input:", INPUT_H5AD)
print("Output:", OUTPUT_H5AD)

In [ ]:
def matrix_values(matrix):
    return matrix.data if sp.issparse(matrix) else np.asarray(matrix).ravel()


def assert_raw_counts(matrix, label: str):
    values = matrix_values(matrix)
    if values.size == 0:
        raise ValueError(f"{label} contains no nonzero values.")
    if not np.isfinite(values).all():
        raise ValueError(f"{label} contains non-finite values.")
    if np.any(values < 0):
        raise ValueError(f"{label} contains negative values.")
    if not np.equal(values, np.floor(values)).all():
        raise ValueError(f"{label} is not an integer raw-count matrix.")


def supported_scrublet_parameters() -> dict:
    requested = {
        "expected_doublet_rate": EXPECTED_DOUBLET_RATE,
        "sim_doublet_ratio": SIM_DOUBLET_RATIO,
        "n_prin_comps": N_PCS,
        "random_state": RANDOM_SEED,
    }
    signature = inspect.signature(sc.pp.scrublet)
    supported = set(signature.parameters)
    missing = sorted(set(requested) - supported)
    if missing:
        raise RuntimeError(
            "This Scanpy version does not support the Scrublet "
            f"parameters required by the recorded analysis: {missing}"
        )
    return requested

In [ ]:
adata = sc.read_h5ad(INPUT_H5AD)

if adata.shape != EXPECTED_INPUT_SHAPE:
    raise AssertionError(
        f"Expected input shape {EXPECTED_INPUT_SHAPE}, observed {adata.shape}"
    )
if SAMPLE_COLUMN not in adata.obs:
    raise KeyError(f"adata.obs[{SAMPLE_COLUMN!r}] is required.")
if "counts" not in adata.layers:
    raise KeyError("The post-QC object is missing layers['counts'].")

assert_raw_counts(adata.layers["counts"], 'adata.layers["counts"]')

for column in (
    "doublet_score",
    "predicted_doublet",
    "doublet_check_status",
):
    if column in adata.obs:
        del adata.obs[column]

adata.obs["doublet_score"] = np.nan
adata.obs["predicted_doublet"] = pd.Series(
    pd.NA,
    index=adata.obs_names,
    dtype="boolean",
)
adata.obs["doublet_check_status"] = pd.Categorical(
    ["not_checked"] * adata.n_obs,
    categories=["not_checked", "checked", "failed"],
)

sample_counts = adata.obs[SAMPLE_COLUMN].value_counts()
print(sample_counts)

In [ ]:
scrublet_parameters = supported_scrublet_parameters()
rng = np.random.default_rng(RANDOM_SEED)
failures = []
run_rows = []

grouped_indices = adata.obs.groupby(
    SAMPLE_COLUMN,
    observed=True,
    sort=True,
).indices

for sample_id, integer_indices in grouped_indices.items():
    sample_cell_ids = adata.obs_names[
        np.asarray(integer_indices, dtype=int)
    ].to_numpy(copy=True)
    n_cells = len(sample_cell_ids)

    print(f"\nSample: {sample_id} | cells: {n_cells}")

    if n_cells < MIN_CELLS_PER_SAMPLE:
        print("Skipping: fewer than 500 cells.")
        run_rows.append(
            {
                "sample": sample_id,
                "chunk": 0,
                "cells": n_cells,
                "status": "not_checked_too_few_cells",
                "error": "",
            }
        )
        continue

    if n_cells > MAX_CELLS_PER_RUN:
        rng.shuffle(sample_cell_ids)
        chunks = np.array_split(
            sample_cell_ids,
            int(np.ceil(n_cells / MAX_CELLS_PER_RUN)),
        )
    else:
        chunks = [sample_cell_ids]

    for chunk_number, chunk_ids in enumerate(chunks, start=1):
        print(
            f"  Running chunk {chunk_number}/{len(chunks)} | "
            f"cells: {len(chunk_ids)}"
        )

        sample_chunk = adata[chunk_ids, :].copy()
        sample_chunk.X = sample_chunk.layers["counts"].copy()

        if sp.issparse(sample_chunk.X):
            sample_chunk.X = sample_chunk.X.tocsr().astype(
                np.float32,
                copy=False,
            )
        else:
            sample_chunk.X = sp.csr_matrix(
                np.asarray(sample_chunk.X, dtype=np.float32)
            )

        assert_raw_counts(
            sample_chunk.X,
            f"{sample_id}, chunk {chunk_number}",
        )

        # Remove fields not required by Scrublet to reduce peak memory.
        sample_chunk.layers.clear()
        sample_chunk.obsm.clear()
        sample_chunk.varm.clear()
        sample_chunk.uns.clear()
        sample_chunk.obsp.clear()
        sample_chunk.varp.clear()
        sample_chunk.raw = None

        try:
            sc.pp.scrublet(
                sample_chunk,
                **scrublet_parameters,
            )

            required_columns = {
                "doublet_score",
                "predicted_doublet",
            }
            missing = (
                required_columns
                - set(sample_chunk.obs.columns)
            )
            if missing:
                raise RuntimeError(
                    f"Scrublet did not create required columns: {missing}"
                )

            adata.obs.loc[
                sample_chunk.obs_names,
                "doublet_score",
            ] = sample_chunk.obs["doublet_score"].to_numpy()

            adata.obs.loc[
                sample_chunk.obs_names,
                "predicted_doublet",
            ] = (
                sample_chunk.obs["predicted_doublet"]
                .astype(bool)
                .to_numpy()
            )

            adata.obs.loc[
                sample_chunk.obs_names,
                "doublet_check_status",
            ] = "checked"

            predicted = int(
                sample_chunk.obs["predicted_doublet"].sum()
            )
            run_rows.append(
                {
                    "sample": sample_id,
                    "chunk": chunk_number,
                    "cells": len(chunk_ids),
                    "status": "checked",
                    "predicted_doublets": predicted,
                    "error": "",
                }
            )
            print(
                sample_chunk.obs[
                    "predicted_doublet"
                ].value_counts()
            )

        except Exception as exc:
            adata.obs.loc[
                sample_chunk.obs_names,
                "doublet_check_status",
            ] = "failed"
            failures.append(
                {
                    "sample": sample_id,
                    "chunk": chunk_number,
                    "error": repr(exc),
                }
            )
            run_rows.append(
                {
                    "sample": sample_id,
                    "chunk": chunk_number,
                    "cells": len(chunk_ids),
                    "status": "failed",
                    "predicted_doublets": np.nan,
                    "error": repr(exc),
                }
            )

        del sample_chunk
        gc.collect()

run_table = pd.DataFrame(run_rows)
run_table.to_csv(
    AUDIT_DIR / "GSE182109_scrublet_run_log.tsv",
    sep="\t",
    index=False,
)

if failures:
    raise RuntimeError(
        "Scrublet failed for one or more sample chunks. "
        "See GSE182109_scrublet_run_log.tsv. "
        f"Failures: {failures}"
    )

In [ ]:
predicted_doublet = adata.obs["predicted_doublet"]

n_doublets = int((predicted_doublet == True).sum())
n_singlets = int((predicted_doublet == False).sum())
n_not_checked = int(predicted_doublet.isna().sum())

print("Predicted doublets:", n_doublets)
print("Predicted singlets:", n_singlets)
print("Not checked:", n_not_checked)
print(adata.obs["doublet_score"].describe())

historical_comparison = {
    "observed_predicted_doublets": n_doublets,
    "historical_predicted_doublets": EXPECTED_PREDICTED_DOUBLETS,
    "observed_predicted_singlets": n_singlets,
    "historical_predicted_singlets": EXPECTED_PREDICTED_SINGLETS,
    "observed_not_checked": n_not_checked,
    "historical_not_checked": EXPECTED_NOT_CHECKED,
}

if (
    n_doublets != EXPECTED_PREDICTED_DOUBLETS
    or n_singlets != EXPECTED_PREDICTED_SINGLETS
    or n_not_checked != EXPECTED_NOT_CHECKED
):
    print(
        "WARNING: Scrublet calls differ from the historical run. "
        "Review package versions and the per-sample run log."
    )

historical_comparison

In [ ]:
per_sample_summary = (
    adata.obs.groupby(SAMPLE_COLUMN, observed=True)
    .agg(
        n_cells=("predicted_doublet", "size"),
        n_doublets=(
            "predicted_doublet",
            lambda values: int((values == True).sum()),
        ),
        n_singlets=(
            "predicted_doublet",
            lambda values: int((values == False).sum()),
        ),
        n_not_checked=(
            "predicted_doublet",
            lambda values: int(values.isna().sum()),
        ),
        mean_doublet_score=("doublet_score", "mean"),
        median_doublet_score=("doublet_score", "median"),
        maximum_doublet_score=("doublet_score", "max"),
    )
    .reset_index()
)

checked_denominator = (
    per_sample_summary["n_doublets"]
    + per_sample_summary["n_singlets"]
)
per_sample_summary["doublet_percent_checked"] = np.where(
    checked_denominator > 0,
    100
    * per_sample_summary["n_doublets"]
    / checked_denominator,
    np.nan,
)

per_sample_summary.to_csv(
    AUDIT_DIR / "GSE182109_scrublet_summary_by_sample.tsv",
    sep="\t",
    index=False,
)

adata.obs[
    [
        SAMPLE_COLUMN,
        "doublet_score",
        "predicted_doublet",
        "doublet_check_status",
    ]
].to_csv(
    AUDIT_DIR / "GSE182109_scrublet_per_cell_scores.tsv.gz",
    sep="\t",
    compression="gzip",
)

per_sample_summary.sort_values(
    "doublet_percent_checked",
    ascending=False,
)

In [ ]:
checked_scores = adata.obs["doublet_score"].dropna().to_numpy()

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(checked_scores, bins=80, edgecolor="black")
ax.set_xlabel("Scrublet doublet score")
ax.set_ylabel("Cells")
ax.set_title("GSE182109 post-QC Scrublet scores")
fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "GSE182109_scrublet_score_histogram.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close(fig)

In [ ]:
doublet_summary = {
    "dataset_id": "DS3_GSE182109",
    "geo_accession": "GSE182109",
    "scope": "sample-aware post-QC diagnostic",
    "method": "scanpy.pp.scrublet",
    "sample_column": SAMPLE_COLUMN,
    "parameters": scrublet_parameters,
    "minimum_cells_per_sample": MIN_CELLS_PER_SAMPLE,
    "maximum_cells_per_run": MAX_CELLS_PER_RUN,
    "input_cells": int(adata.n_obs),
    "input_genes": int(adata.n_vars),
    "predicted_doublets": n_doublets,
    "predicted_singlets": n_singlets,
    "not_checked": n_not_checked,
    "doublets_removed_from_main_analysis": False,
    "main_analysis_cells": int(adata.n_obs),
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "output_h5ad": str(OUTPUT_H5AD),
}

adata.uns["doublet_detection_provenance"] = {
    "scope": "sample_aware_post_QC_diagnostic",
    "method": "scanpy.pp.scrublet",
    "doublets_removed_from_main_analysis": "false",
    "predicted_doublets": n_doublets,
    "predicted_singlets": n_singlets,
    "not_checked": n_not_checked,
    "parameters_json": json.dumps(
        scrublet_parameters,
        sort_keys=True,
    ),
}

with open(
    AUDIT_DIR / "GSE182109_scrublet_summary.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(doublet_summary, handle, indent=2)

print(doublet_summary)

In [ ]:
if adata.shape != EXPECTED_INPUT_SHAPE:
    raise AssertionError(
        "The doublet diagnostic unexpectedly changed the object shape: "
        f"{adata.shape}"
    )

temporary_output = OUTPUT_H5AD.with_suffix(".tmp.h5ad")
if temporary_output.exists():
    temporary_output.unlink()

adata.write_h5ad(temporary_output, compression="gzip")
temporary_output.replace(OUTPUT_H5AD)

with sc.read_h5ad(OUTPUT_H5AD, backed="r") as check:
    if check.shape != EXPECTED_INPUT_SHAPE:
        raise AssertionError(
            f"Saved-file validation failed: observed {check.shape}"
        )
    for column in (
        "doublet_score",
        "predicted_doublet",
        "doublet_check_status",
    ):
        if column not in check.obs:
            raise AssertionError(
                f"Saved output is missing adata.obs[{column!r}]."
            )

print("Saved and validated:", OUTPUT_H5AD)